# Emergency Classical Export

Run this if the normal no-RL freeze/export notebooks failed and you need a classical `agent.zip` immediately.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import hashlib
import time

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/search.py").is_file():
    raise FileNotFoundError(f"Project files missing from {PROJECT_ROOT}")

out = PROJECT_ROOT / "exports/emergency_classical"
cand = out / f"candidate_{int(time.time())}"
runtime = cand / "chess_runtime"
runtime.mkdir(parents=True, exist_ok=True)

files = [
    "__init__.py",
    "piece_tables.py",
    "classical_evaluation.py",
    "environment.py",
    "time_management.py",
    "transposition.py",
    "search.py",
]

for name in files:
    shutil.copy2(PROJECT_ROOT / "chess_rl" / name, runtime / name)

shutil.copy2(PROJECT_ROOT / "templates/classical_agent.py", cand / "agent.py")

config = {
    "model_version": "emergency_classical",
    "search": {
        "max_depth": 1,
        "quiescence_depth": 0,
        "value_eval_mix": 0.0,
        "policy_ordering": False,
        "time_budget_fraction": 0.05,
        "max_budget_ms": 100,
        "transposition_table_size": 100000,
        "evaluation_weights": {
            "material": 1.0,
            "piece_square": 1.0,
            "activity": 2.0,
            "bishop_pair": 35.0,
            "doubled_pawns": -12.0,
            "isolated_pawns": -10.0,
            "passed_pawns": 20.0,
            "passed_pawn_advance": 8.0,
        },
    },
}

(cand / "config.json").write_text(json.dumps(config, indent=2))
(cand / "model_manifest.json").write_text(
    json.dumps({"workflow": "emergency_classical", "candidate": cand.name}, indent=2)
)

zip_path = out / "agent.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in cand.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(cand))

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    assert "agent.py" in names
    assert "config.json" in names

print("UPLOAD THIS:", zip_path)
print("SHA256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())
print("Contents:", names)